In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mpltransforms

import seaborn as sns
import os
import datetime
import pickle

In [3]:
if os.getcwd().endswith('/notebooks/figures'):
    os.chdir('../../')

In [4]:
from src.dataproc import utils, ccutils

### Supporting functions

In [5]:
# updated range on 7/14/2026
def szred_to_responder(rednum):
    if rednum > 0.9:
        return 'SR'
    elif rednum < 0.5:
        return 'NR'
    else:
        return 'IR'

### Stimulation

In [6]:
fname_format = 'data/RNSdata/NPEhistograms/%s_Histogram_Hourly.csv'
patcodes_path = 'data/RNSdata/Outcome/Hippo_outcomes_postprocessed_2025April08.csv'
longi_patcodes_path = 'data/RNSdata/Outcome/Hippo_longitudinal_outcomes_postprocessed_2025April08.csv'
lsdf_path = 'data/processed/lscycles/plv_postproc_interval/%s.csv'  # interval or not?
plvdf_path = 'data/processed/ecogfeatures/plv_postproc/%s_trigger-Scheduled.csv'

In [7]:
outcome_type = 'wtavg_perc_szred'

patcodesdf = pd.read_csv(patcodes_path)

patcodesdf['sz_red'] = patcodesdf[outcome_type].apply(lambda x: -x/100 if x < 0 else 0)
patcodesdf['seizure_rate'] = patcodesdf['baseline_seizure_rate'] * (1 - patcodesdf['sz_red'])
patcodesdf['Responder_Type'] = patcodesdf['sz_red'].apply(lambda x: szred_to_responder(x))

patient_list = ccutils.final_patient_list()

patoutcome_cols = ['npe_code', 'sz_red', 'log_baseline_seizure_rate', 'log_seizure_rate']

### only consider the subset of patients in the final list
lgpatcodesdf = pd.read_csv(longi_patcodes_path) # longitudinal outcomes
lgpatcodesdf = lgpatcodesdf.loc[lgpatcodesdf['npe_code'].isin(patient_list)].copy()

In [8]:
### number of patients
print(f'Number of patients: {len(patient_list)}')

Number of patients: 38


In [9]:
### info about ECoG scheduled recordings

npe0041_date_format = "%m/%d/%y %H:%M"

countdf = {key: [] for key in ['npecode', 'num_clips', 'num_recdays', 'num_stimdays']}

for patid in patient_list[:]:
    
    plvdf = pd.read_csv(plvdf_path % (patid))
    
    # day of ECoG Scheuled recording from the start of implantation
    implant_date = datetime.datetime.fromisoformat(patcodesdf.loc[patcodesdf['npe_code']==patid]['implant_date'].item())
    utils.tfromstart_days(plvdf, start_date=implant_date)  # time from the date of implantation

    # days of hourly stim counts from the start of implantation
    df = pd.read_csv(fname_format %(patid))
    if patid == 'NPE0041': # update the formatting of time bins for this patient
        df['Region start time'] = df['Region start time'].apply(lambda x: datetime.datetime.strptime(x, npe0041_date_format).isoformat(sep=' '))
    df['Local_hour'] = df['Region start time'].apply(lambda x: datetime.datetime.fromisoformat(x).hour)
    df['npe_code'] = patid
    utils.tfromstart_days(df, datecol='Region start time', start_date=implant_date)

    # store results
    countdf['npecode'].append(patid)
    countdf['num_clips'].append( plvdf.shape[0] )
    countdf['num_recdays'].append(np.round(plvdf['tfromstart_days'].max()))  # number of days
    countdf['num_stimdays'].append( np.round( df['tfromstart_days'].max() ) ) 
    
# convert to arrays
countdf = pd.DataFrame(countdf)
countdf['num_rnsdays'] = countdf.apply(lambda x: int(max(x['num_recdays'], x['num_stimdays'])), axis=1)
countdf['num_rnsmos'] = countdf['num_rnsdays'].apply(lambda x: np.round(x/30,1))

In [10]:
print(f"Total #scheduled recordings = {countdf['num_clips'].sum()}")
print(f"Total #days recordings = {countdf['num_rnsdays'].sum()}")

Total #scheduled recordings = 99349
Total #days recordings = 66732


In [11]:
### number of self-reported seizure counts
print('Number of visits =', lgpatcodesdf.shape[0])
print("Mean visits per patient = %0.1f \pm %0.1f"%(lgpatcodesdf['npe_code'].value_counts().mean(), 
                                                   lgpatcodesdf['npe_code'].value_counts().std()))
print("Mean interval = %0.1f \pm %0.1f"%(lgpatcodesdf['visit_interval'].mean(), 
                                         lgpatcodesdf['visit_interval'].std()))

Number of visits = 145
Mean visits per patient = 3.8 \pm 2.1
Mean interval = 448.1 \pm 404.4


In [12]:
### number of self-reported seizure counts
print('Number of visits =', lgpatcodesdf.shape[0])
print("Median visits per patient = %0.1f (IQR %0.1f - %0.1f)"%(lgpatcodesdf['npe_code'].value_counts().median(), 
                                                               lgpatcodesdf['npe_code'].value_counts().quantile(0.25), 
                                                               lgpatcodesdf['npe_code'].value_counts().quantile(0.75)) )
print("Median interval = %0.1f (IQR %0.1f - %0.1f)"%(lgpatcodesdf['visit_interval'].median(), 
                                                     lgpatcodesdf['visit_interval'].quantile(0.25), 
                                                     lgpatcodesdf['visit_interval'].quantile(0.75)))

Number of visits = 145
Median visits per patient = 3.0 (IQR 2.0 - 5.0)
Median interval = 259.0 (IQR 140.0 - 628.0)


In [13]:
countdf = countdf.merge(patcodesdf[['npe_code', 'sz_red']], how='left', left_on='npecode', right_on='npe_code')

In [14]:
countdf['sz_red_str'] = countdf['sz_red'].apply(lambda x: '%0.2f'%(x))

In [15]:
save_flag = False
if save_flag:
    countdf[['npecode', 'num_rnsdays', 'num_clips', 'sz_red_str']].to_csv('output/circpaper_figures/tableS1_stats.csv', index=False)